# Advanced Chunking Methods

In [1]:
from gitsource import GithubRepositoryDataReader
from gitsource.chunking import sliding_window
from pydantic import BaseModel
from openai import OpenAI
from dotenv import load_dotenv
import os
import re

In [2]:
reader = GithubRepositoryDataReader(
    repo_name="docs",
    repo_owner="evidentlyai",
    allowed_extensions={"md", "mdx"}
)

files = reader.read()
parsed_docs = [doc.parse() for doc in files]

print(f"Loaded {len(parsed_docs)} documents")

Loaded 95 documents


In [3]:
parsed_docs[10]

{'title': 'Output formats',
 'description': 'How to export the evaluation results.',
 'content': 'You can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python

## Token-based Chunking

In [4]:
texts = []
for doc in parsed_docs:
    title = doc.get('title', '')
    description = doc.get('description', '')
    content = doc.get('content', '')
    text = title + ' ' + description + ' ' + content
    texts.append(text.strip())

In [5]:
text = parsed_docs[45]['content']
words = text.split()

chunk_size = 500
step = 250
chunks = sliding_window(words, size=chunk_size, step=step)

print(f"Created {len(chunks)} chunks")

Created 11 chunks


In [6]:
first_chunk = " ".join(chunks[0]['content'])
print(chunks[1]['content'])
print(parsed_docs[1]['openapi'])

['import', 'pandas', 'as', 'pd', 'from', 'evidently.future.datasets', 'import', 'Dataset', 'from', 'evidently.future.datasets', 'import', 'DataDefinition', 'from', 'evidently.future.datasets', 'import', 'Descriptor', 'from', 'evidently.future.descriptors', 'import', '*', 'from', 'evidently.future.report', 'import', 'Report', 'from', 'evidently.future.presets', 'import', 'TextEvals', 'from', 'evidently.future.metrics', 'import', '*', 'from', 'evidently.future.tests', 'import', '*', 'from', 'evidently.features.llm_judge', 'import', 'BinaryClassificationPromptTemplate', '```', 'To', 'connect', 'to', 'Evidently', 'Cloud:', '```python', 'from', 'evidently.ui.workspace.cloud', 'import', 'CloudWorkspace', '```', '**Optional.**', 'To', 'create', 'monitoring', 'panels', 'as', 'code:', '```python', 'from', 'evidently.ui.dashboards', 'import', 'DashboardPanelPlot', 'from', 'evidently.ui.dashboards', 'import', 'DashboardPanelTestSuite', 'from', 'evidently.ui.dashboards', 'import', 'DashboardPanelT

## Paragraph-based Chunking

In [7]:
text = parsed_docs[45]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())
print(f"Found {len(paragraphs)} paragraphs")
print(paragraphs[:3])

Found 153 paragraphs
['In this tutorial, you will learn how to perform regression testing for LLM outputs.', 'You can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.', "<Info>\n  **This example uses Evidently Cloud.** You'll run evals in Python and upload them. You can also skip the upload and view Reports locally. For self-hosted, replace `CloudWorkspace` with `Workspace`.\n</Info>"]


## Section-based Chunking

In [8]:
header_pattern = r"^(#{2} )(.+)$"
pattern = re.compile(header_pattern, re.MULTILINE)

In [9]:
example_text = """
# Introduction

This is the intro section.

## Getting Started

Here's how to get started.

## Advanced Usage

Here are advanced topics
""".strip()

parts = pattern.split(example_text)

print(f"Number of parts: {len(parts)}")
print(parts)

Number of parts: 7
['# Introduction\n\nThis is the intro section.\n\n', '## ', 'Getting Started', "\n\nHere's how to get started.\n\n", '## ', 'Advanced Usage', '\n\nHere are advanced topics']


In [10]:
sections = []

for i in range(1, len(parts), 3):
    header = parts[i] + parts[i+1]
    header = header.strip()

    content = ""

    if i+2 < len(parts):
        content = parts[i+2].strip()
    
    if content:
        section = f'{header}\n\n{content}'
    else:
        section = header

    sections.append(section)

print(f"Found {len(sections)} sections")

for section in sections:
    print("-----")
    print(section)

Found 2 sections
-----
## Getting Started

Here's how to get started.
-----
## Advanced Usage

Here are advanced topics


In [11]:
def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)
    parts = pattern.split(text)
    
    sections = []

    for i in range(1, len(parts), 3):
        header = parts[i] + parts[i+1]
        header = header.strip()

        content = ""

        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header

        sections.append(section)
    
    return sections

In [12]:
sections = split_markdown_by_level(text, level=2)
print(f"Found {len(sections)} sections")

Found 8 sections


## Combined

In [13]:
sections = split_markdown_by_level(text, level=2)
chunks = sliding_window(sections, 3, 2)

combined_chunks = []
for chunk in chunks:
    chunk_text = "\n\n".join(chunk['content'])
    combined_chunks.append(chunk_text)

print(f"Created {len(combined_chunks)} chunks")
print(combined_chunks[0] + "....")

Created 4 chunks
## 1. Installation and Imports

Install Evidently:

```python
pip install evidently[llm] 
```

Import the required modules:

```python
import pandas as pd
from evidently.future.datasets import Dataset
from evidently.future.datasets import DataDefinition
from evidently.future.datasets import Descriptor
from evidently.future.descriptors import *
from evidently.future.report import Report
from evidently.future.presets import TextEvals
from evidently.future.metrics import *
from evidently.future.tests import *

from evidently.features.llm_judge import BinaryClassificationPromptTemplate
```

To connect to Evidently Cloud:

```python
from evidently.ui.workspace.cloud import CloudWorkspace
```

**Optional.** To create monitoring panels as code:

```python
from evidently.ui.dashboards import DashboardPanelPlot
from evidently.ui.dashboards import DashboardPanelTestSuite
from evidently.ui.dashboards import DashboardPanelTestSuiteCounter
from evidently.ui.dashboards import TestSu

In [14]:
for chunk in chunks:
    print(len(chunk['content']))

3
3
3
2


## Intelligent Chunking with LLMs

In [15]:
load_dotenv()
openai_key = os.environ.get("OPENAI_API_KEY")
client = OpenAI(api_key=openai_key)

In [16]:
chunking_instructions = """
Split the provided document into logical sections that make sense for a Q&A system.
Each section should be self-contained and cover a specific topic or concept.
You should copy each section word for word from the original text. Do not add or omit any word or character.
Sections should relatively large (3000 - 5000 characters)
""".strip()

print(chunking_instructions)

Split the provided document into logical sections that make sense for a Q&A system.
Each section should be self-contained and cover a specific topic or concept.
You should copy each section word for word from the original text. Do not add or omit any word or character.
Sections should relatively large (3000 - 5000 characters)


In [17]:
class Section(BaseModel):
    title: str
    markdown: str

class Document(BaseModel):
    title: str
    sections: list[Section]

In [18]:
def structured_llm(client, user_prompt, output_type, system_instructions=None, model="gpt-4o-mini"):
    messages = []

    if system_instructions:
        messages.append({
            "role":"system",
            "content":system_instructions
        })
    
    messages.append({
        "role":"user",
        "content":user_prompt
    })

    response = client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed

In [19]:
chunks = structured_llm(client, text, output_type=Document, system_instructions=chunking_instructions)
chunks

Document(title='Regression Testing for LLM Outputs', sections=[Section(title='Introduction to Regression Testing for LLM Outputs', markdown="In this tutorial, you will learn how to perform regression testing for LLM outputs.\n\nYou can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.\n\n<Info>\n  **This example uses Evidently Cloud.** You'll run evals in Python and upload them. You can also skip the upload and view Reports locally. For self-hosted, replace `CloudWorkspace` with `Workspace`.\n</Info>\n\n# Tutorial scope\n\nHere's what we'll do:\n\n* **Create a toy dataset**. Build a small Q&A dataset with answers and reference responses.\n\n* **Get new answers**. Imitate generating new answers to the same question.\n\n* **Create and run a Report with Tests**. Compare the answers using

In [20]:
print(text)

In this tutorial, you will learn how to perform regression testing for LLM outputs.

You can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.

<Info>
  **This example uses Evidently Cloud.** You'll run evals in Python and upload them. You can also skip the upload and view Reports locally. For self-hosted, replace `CloudWorkspace` with `Workspace`.
</Info>

# Tutorial scope

Here's what we'll do:

* **Create a toy dataset**. Build a small Q&A dataset with answers and reference responses.

* **Get new answers**. Imitate generating new answers to the same question.

* **Create and run a Report with Tests**. Compare the answers using LLM-as-a-judge to evaluate length, correctness and style consistency.

* **Build a monitoring Dashboard**. Get plots to track the results of Tests over time

In [21]:
print(chunks.sections[0].markdown)

In this tutorial, you will learn how to perform regression testing for LLM outputs.

You can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.

<Info>
  **This example uses Evidently Cloud.** You'll run evals in Python and upload them. You can also skip the upload and view Reports locally. For self-hosted, replace `CloudWorkspace` with `Workspace`.
</Info>

# Tutorial scope

Here's what we'll do:

* **Create a toy dataset**. Build a small Q&A dataset with answers and reference responses.

* **Get new answers**. Imitate generating new answers to the same question.

* **Create and run a Report with Tests**. Compare the answers using LLM-as-a-judge to evaluate length, correctness and style consistency.

* **Build a monitoring Dashboard**. Get plots to track the results of Tests over time